# Lab 12 — Matrix Denoising via SVD
**Linear Algebra · UATX**

We observe $M = A_{\text{signal}} + E$ where $A_{\text{signal}}$ is low-rank (the signal) and $E$ is Gaussian noise. SVD truncation $\hat{A}_k = \sum_{i=1}^k \sigma_i u_i v_i^\top$ recovers the signal.

**Marchenko–Pastur threshold:** for $E \in \mathbb{R}^{m \times n}$ with i.i.d. $\mathcal{N}(0,\sigma^2)$ entries ($m \leq n$), all noise singular values lie below $\sigma(\sqrt{m}+\sqrt{n})$ with high probability.

**Tasks**
1. Build a rank-3 signal $+$ Gaussian noise; visualise the singular-value spectrum.
2. Truncate at $k=1,\ldots,5$ and find the optimal $k$.
3. Implement an elbow-detection rank estimator and compare it to the MP threshold.
4. Denoise a real image; compute PSNR before and after.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import imageio.v3 as iio
np.random.seed(99)

## §1  Synthetic Low-Rank Signal + Noise

In [ ]:
m, n = 100, 200
rank_true = 3
sigma_noise = 0.5

# Build rank-3 signal with prescribed singular values
U0, _ = np.linalg.qr(np.random.randn(m, rank_true))
V0, _ = np.linalg.qr(np.random.randn(n, rank_true))
sigma_signal = np.array([10., 5., 2.])
A_signal = (U0 * sigma_signal) @ V0.T

# Add Gaussian noise
E = sigma_noise * np.random.randn(m, n)
M = A_signal + E

# SVD of noisy matrix
U, s, Vt = np.linalg.svd(M, full_matrices=False)

# Marchenko-Pastur threshold (known sigma)
mp_threshold = sigma_noise * (np.sqrt(m) + np.sqrt(n))
print(f'Marchenko-Pastur threshold: sigma*(sqrt(m)+sqrt(n)) = {sigma_noise}*({np.sqrt(m):.2f}+{np.sqrt(n):.2f}) = {mp_threshold:.2f}')
print(f'True signal singular values: {sigma_signal}')
print(f'Noise floor visible below: {mp_threshold:.2f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# All singular values
axes[0].plot(s, 'o-', ms=4, alpha=0.7, label='$\\sigma_i(M)$')
axes[0].axhline(mp_threshold, color='red', lw=2, ls='--', label=f'MP threshold = {mp_threshold:.1f}')
for i, sv in enumerate(sigma_signal):
    axes[0].axhline(sv, color='green', lw=1, ls=':', alpha=0.7)
axes[0].set_xlabel('Index'); axes[0].set_ylabel('Singular value')
axes[0].set_title('Singular values of $M = A_{\\rm signal} + E$')
axes[0].legend(); axes[0].set_xlim(-1, 30)

# First 20 singular values
axes[1].bar(range(20), s[:20], alpha=0.7)
axes[1].axhline(mp_threshold, color='red', lw=2, ls='--', label='MP threshold')
axes[1].set_xlabel('Index'); axes[1].set_ylabel('Singular value')
axes[1].set_title('First 20 singular values')
axes[1].legend()

plt.tight_layout(); plt.show()
print(f'Singular values above MP threshold: {np.sum(s > mp_threshold)}  (true rank = {rank_true})')

## §2  SVD Truncation and Recovery Error

In [ ]:
def truncate_svd(U, s, Vt, k):
    """Rank-k approximation."""
    return (U[:, :k] * s[:k]) @ Vt[:k, :]

fro_errors = []
for k in range(1, 8):
    Ak = truncate_svd(U, s, Vt, k)
    err = np.linalg.norm(Ak - A_signal, 'fro')
    fro_errors.append(err)
    print(f'k={k}: ||A_k - A_signal||_F = {err:.4f}')

opt_k = np.argmin(fro_errors) + 1
print(f'\nOptimal k = {opt_k}  (true rank = {rank_true})')

plt.figure(figsize=(6, 4))
plt.plot(range(1, 8), fro_errors, 'o-', ms=8)
plt.axvline(opt_k, color='red', ls='--', label=f'optimal k={opt_k}')
plt.xlabel('Truncation rank $k$'); plt.ylabel('$\\|\\hat{A}_k - A_{\\rm signal}\\|_F$')
plt.title('Recovery error vs truncation rank'); plt.legend(); plt.grid(True)
plt.tight_layout(); plt.show()

## §3  Rank Estimation: Marchenko–Pastur vs Elbow Detection

In [ ]:
def mp_rank(s, m, n, sigma):
    """Marchenko-Pastur rank estimate (known sigma)."""
    threshold = sigma * (np.sqrt(m) + np.sqrt(n))
    return int(np.sum(s > threshold))


def elbow_rank(s, n_candidates=None):
    """
    Elbow method: find the index maximising perpendicular distance from
    the line connecting (0, s[0]) to (len(s)-1, s[-1]).
    """
    if n_candidates is None:
        n_candidates = min(50, len(s))
    s_sub = s[:n_candidates].copy()
    N = len(s_sub)
    x = np.arange(N, dtype=float)
    # Line from first to last point
    p1 = np.array([0, s_sub[0]])
    p2 = np.array([N-1, s_sub[-1]])
    d = p2 - p1
    d_norm = d / np.linalg.norm(d)
    # Perpendicular distances
    dists = []
    for i in range(N):
        p = np.array([x[i], s_sub[i]])
        t = np.dot(p - p1, d_norm)
        proj = p1 + t * d_norm
        dists.append(np.linalg.norm(p - proj))
    return int(np.argmax(dists)) + 1   # +1 to convert to rank


# Test on our synthetic data (sigma known)
r_mp = mp_rank(s, m, n, sigma_noise)
r_elbow = elbow_rank(s)
print(f'True rank:             {rank_true}')
print(f'MP threshold rank:     {r_mp}')
print(f'Elbow rank:            {r_elbow}')

# Plot elbow
fig, ax = plt.subplots(figsize=(7, 4))
idx = np.arange(min(30, len(s)))
ax.plot(idx, s[:len(idx)], 'o-', ms=5)
ax.axvline(r_mp - 1,    color='red',    ls='--', lw=2, label=f'MP rank = {r_mp}')
ax.axvline(r_elbow - 1, color='purple', ls=':',  lw=2, label=f'Elbow rank = {r_elbow}')
ax.set_xlabel('Index'); ax.set_ylabel('$\\sigma_i$')
ax.set_title('Rank estimation on singular-value spectrum')
ax.legend(); ax.grid(True)
plt.tight_layout(); plt.show()

In [ ]:
# Sweep over multiple noise levels
noise_levels = [0.1, 0.3, 0.5, 1.0, 2.0]
results = []
for sig in noise_levels:
    E_i = sig * np.random.randn(m, n)
    M_i = A_signal + E_i
    U_i, s_i, Vt_i = np.linalg.svd(M_i, full_matrices=False)
    r_mp_i    = mp_rank(s_i, m, n, sig)
    r_elbow_i = elbow_rank(s_i)
    results.append((sig, r_mp_i, r_elbow_i))
    print(f'sigma={sig:.1f}: MP rank={r_mp_i}, elbow rank={r_elbow_i}  (true={rank_true})')

## §4  Denoising a Real Image

In [ ]:
def psnr(original, reconstructed, max_val=1.0):
    """Peak signal-to-noise ratio (dB). Higher is better."""
    mse = np.mean((original - reconstructed)**2)
    if mse < 1e-15:
        return float('inf')
    return 10 * np.log10(max_val**2 / mse)


# Load a standard test image (grayscale)
img = iio.imread('imageio:camera.png').astype(float) / 255.0

# Add Gaussian noise
noise_sigma = 0.08
np.random.seed(5)
noisy = np.clip(img + noise_sigma * np.random.randn(*img.shape), 0, 1)

# SVD of noisy image
U_img, s_img, Vt_img = np.linalg.svd(noisy, full_matrices=False)
m_img, n_img = img.shape

# Estimate noise sigma from median singular value (robust estimator)
s_med = np.median(s_img)
sigma_est = s_med / (0.6745 * np.sqrt(max(m_img, n_img)))
print(f'Image size: {img.shape}')
print(f'Added noise sigma: {noise_sigma:.3f}')
print(f'Estimated sigma:   {sigma_est:.3f}')

# Rank estimates
r_mp_img    = mp_rank(s_img, m_img, n_img, noise_sigma)   # using known sigma
r_elbow_img = elbow_rank(s_img, n_candidates=100)
print(f'\nMP rank (known sigma):   {r_mp_img}')
print(f'Elbow rank:              {r_elbow_img}')

In [ ]:
# Reconstruct at different ranks
rank_choices = [r_elbow_img, r_mp_img, r_elbow_img * 2]
fig, axes = plt.subplots(1, len(rank_choices) + 2, figsize=(16, 4))

axes[0].imshow(img, cmap='gray', vmin=0, vmax=1)
axes[0].set_title(f'Original\n(PSNR = inf)'); axes[0].axis('off')

axes[1].imshow(noisy, cmap='gray', vmin=0, vmax=1)
axes[1].set_title(f'Noisy\n(PSNR = {psnr(img, noisy):.1f} dB)'); axes[1].axis('off')

for ax, k in zip(axes[2:], rank_choices):
    denoised = truncate_svd(U_img, s_img, Vt_img, k)
    denoised = np.clip(denoised, 0, 1)
    p = psnr(img, denoised)
    ax.imshow(denoised, cmap='gray', vmin=0, vmax=1)
    ax.set_title(f'Rank {k}\n(PSNR = {p:.1f} dB)'); ax.axis('off')

plt.suptitle('SVD Image Denoising')
plt.tight_layout(); plt.show()

In [ ]:
# PSNR curve vs rank
ranks_to_test = list(range(1, min(120, m_img+1), 3))
psnr_vals = []
for k in ranks_to_test:
    rec = np.clip(truncate_svd(U_img, s_img, Vt_img, k), 0, 1)
    psnr_vals.append(psnr(img, rec))

best_k = ranks_to_test[np.argmax(psnr_vals)]
best_psnr = max(psnr_vals)

plt.figure(figsize=(8, 4))
plt.plot(ranks_to_test, psnr_vals, lw=2)
plt.axvline(r_mp_img, color='red', ls='--', label=f'MP threshold (k={r_mp_img})')
plt.axvline(best_k, color='green', ls=':', label=f'Best PSNR k={best_k} ({best_psnr:.1f} dB)')
plt.axhline(psnr(img, noisy), color='gray', ls='--', label=f'Noisy PSNR ({psnr(img,noisy):.1f} dB)')
plt.xlabel('Rank $k$'); plt.ylabel('PSNR (dB)')
plt.title('Denoising quality vs truncation rank')
plt.legend(); plt.grid(True)
plt.tight_layout(); plt.show()